<a href="https://colab.research.google.com/github/Peeyusj/gpt_from_sratch/blob/main/gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - download the dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-26 10:55:56--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-04-26 10:55:56 (22.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# Cell 2 - read it and explore
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print("First 200 characters:")
print(text[:200])

Total characters: 1115394
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [3]:
vocab= sorted(set(text))
print(len(vocab))
print(vocab)

65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
stoi={}
itos={}
for i, ch in enumerate(vocab):
    itos[i]=ch
    stoi[ch]=i

print(stoi)
print(itos)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}
{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i',

In [5]:
def encode(strVal):
  res=[]
  for i in strVal:
    res.append(stoi[i])
  return res

def decode(intVal):
  res=[]
  for i in intVal:
    res.append(itos[i])
  return "".join(res)

print(encode("hello"))
print(decode(encode("hello")))


[46, 43, 50, 50, 53]
hello


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape)
print(data[:10])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [7]:
n = int(0.9 * len(data))

train_data =data[:n]   # everything before n
val_data =data[n:]   # everything from n onwards
print('train_data-',train_data.shape)
print('val_data-',val_data.shape)

train_data- torch.Size([1003854])
val_data- torch.Size([111540])


In [8]:
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # now build x and y using ix
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [9]:
xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)
print("Input batch:")
print(xb)
print("\nTarget batch:")
print(yb)

torch.Size([4, 8])
torch.Size([4, 8])
Input batch:
tensor([[43, 56, 43,  1, 47, 57,  1, 52],
        [53, 58,  1, 44, 53, 56,  1, 58],
        [47, 56, 43, 10,  0, 13, 50, 50],
        [59, 54, 53, 52,  1, 39,  1, 57]])

Target batch:
tensor([[56, 43,  1, 47, 57,  1, 52, 53],
        [58,  1, 44, 53, 56,  1, 58, 46],
        [56, 43, 10,  0, 13, 50, 50,  1],
        [54, 53, 52,  1, 39,  1, 57, 59]])


In [10]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            return logits, None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
            return logits, loss

    def generate(self, idx, max_new_tokens):
      for _ in range(max_new_tokens):
        logits, loss = self(idx)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
      return idx

In [11]:
vocab_size = 65
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.9903, grad_fn=<NllLossBackward0>)


In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(10000):
    xb, yb = get_batch('train')          # step 1: get batch
    logits, loss = model(xb, yb)         # step 2: forward pass
    optimizer.zero_grad()                # step 3: zero gradients
    loss.backward()                      # step 4: backward pass
    optimizer.step()                     # step 5: update weights

    if step % 1000 == 0:
        print(f"step {step}: loss {loss.item():.4f}")

step 0: loss 4.7914
step 1000: loss 4.5733
step 2000: loss 3.6465
step 3000: loss 3.3237
step 4000: loss 2.8616
step 5000: loss 3.0545
step 6000: loss 2.6477
step 7000: loss 2.5887
step 8000: loss 2.5360
step 9000: loss 2.6588


In [13]:
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))


Wh;
BAnzY! y:
Ecinthefsok
IN:Xwhe;
LUf; thingmeearepy p'styon Wifs ite'soupld h.
I hin, Chetat nnethond op h itit hise; fe hy cheismes t
Aiglju w?-ve Yes
PQN RMSTofed ibyoratr, mee myor.
Ann wie:-nute
